In [3]:
import optuna
import inspect
import numpy as np
import pandas as pd
import pandas_ta as ta
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

/Users/habib/Desktop/Projects/FinDRL/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Idea
## It's an DL way to Optimize TA Indicators

> 1. Input: Stock market time sereas data and selected the datas specific field (Ex: Close)
> 2. 

In [3]:
# # Install tuneta
# !pip install tuneta
# # Install scikit-learn
# !pip install scikit-learn

In [4]:
# from tuneta.tune_ta import TuneTA
# import pandas as pd
# from pandas_ta import percent_return
# from sklearn.model_selection import train_test_split
# import yfinance as yf

In [5]:
# # Download data set from yahoo, calculate next day return and split into train and test
# X = yf.download("SPY", period="10y", interval="1d", auto_adjust=True)


In [6]:
# # Check if columns have MultiIndex
# if isinstance(X.columns, pd.MultiIndex):
#     X.columns = ['_'.join(col).strip() for col in X.columns.values]

# X.rename(columns={
#     'Close_SPY': 'Close',
#     'High_SPY': 'High',
#     'Low_SPY': 'Low',
#     'Open_SPY': 'Open',
#     'Volume_SPY': 'Volume'
# }, inplace=True)

# X

In [7]:
# y = percent_return(X.Close, offset=-1)
# y

In [8]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, shuffle=False)

# # Initialize with x cores and show trial results
# tt = TuneTA(n_jobs=6, verbose=True)

# # Optimize indicators
# tt.fit(X_train, y_train,
#     indicators=['all'],
#     ranges=[(4, 30)],
#     trials=100,
#     early_stop=10,
# )

# # Show time duration in seconds per indicator
# tt.fit_times()

# # Show correlation of indicators to target
# tt.report(target_corr=True, features_corr=True)

# # Select features with at most x correlation between each other
# tt.prune(max_inter_correlation=.7)

# # Show correlation of indicators to target and among themselves
# tt.report(target_corr=True, features_corr=True)

# # Add indicators to X_train
# features = tt.transform(X_train)
# X_train = pd.concat([X_train, features], axis=1)

# # Add same indicators to X_test
# features = tt.transform(X_test)
# X_test = pd.concat([X_test, features], axis=1)

In [9]:
# X_test

In [10]:
# X_test

In [28]:
# Backtest Engine with Indicator Parameter Optimization
class IndicatorOptimizer:
    def __init__(self, data, field_map, indicator_func, initial_cash=10000, transaction_cost=0.001):
        self.data = data
        self.field_map = field_map
        self.indicator_func = indicator_func
        self.initial_cash = initial_cash
        self.transaction_cost = transaction_cost
    
    def optimize_parameters(self, param_grid, epochs=20, lookback=60):
        """
        Optimize technical indicator parameters using deep learning.
        :param param_grid: Dictionary of parameter ranges for the indicator.
        :param epochs: Number of epochs for training the DL model.
        :param lookback: Lookback period for LSTM.
        """
        def objective(trial):
            # Sample parameters
            params = {k: trial.suggest_int(k, v[0], v[1]) for k, v in param_grid.items()}
            print(f"Testing parameters: {params}")
            
            # Apply indicator with sampled parameters
            df_with_indicators = self.apply_indicator(params)
            
            # Prepare data for DL model
            X, y, scaler = self.prepare_data(df_with_indicators, lookback)
            split = int(0.8 * len(X))
            X_train, X_test = X[:split], X[split:]
            y_train, y_test = y[:split], y[split:]
            
            # Train and evaluate DL model
            model = self.build_lstm(X_train.shape[1:])
            model.fit(X_train, y_train, batch_size=1, epochs=epochs, verbose=0)
            y_pred = model.predict(X_test).flatten()
            y_test_actual = scaler.inverse_transform([[0] * (X.shape[2] - 1) + [v] for v in y_test])
            y_pred_actual = scaler.inverse_transform([[0] * (X.shape[2] - 1) + [v] for v in y_pred])
            
            # Calculate performance
            sharpe_ratio = self.calculate_sharpe_ratio(y_test_actual, y_pred_actual)
            return -sharpe_ratio  # Optuna minimizes, so return negative of Sharpe
            
        # Optimize using Optuna
        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=50)
        return study.best_params, -study.best_value  # Return the best parameters and performance
    
    def apply_indicator(self, params):
        """Apply the indicator to the data."""
        df = self.data.copy()
        df['Indicator'] = self.indicator_func(df[self.field_map['price']], **params)
        return df
    
    def prepare_data(self, data, lookback):
        """Prepare data for LSTM model."""
        feature_col = ['Indicator', self.field_map['price']]
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(data[feature_col])

        X, y = [], []
        for i in range(lookback, len(scaled_data)):
            X.append(scaled_data[i - lookback:i])
            y.append(scaled_data[i, 1])  # Target: price column

        return np.array(X), np.array(y), scaler
    
    def build_lstm(self, input_shape):
        """Build LSTM model."""
        model = Sequential([
            LSTM(50, return_sequences=True, input_shape=input_shape),
            Dropout(0.2),
            LSTM(50, return_sequences=False),
            Dropout(0.2),
            Dense(25),
            Dense(1)  # Predict a single value
        ])
        model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
        return model

    def calculate_sharpe_ratio(self, actual, predicted):
        """Calculate Sharpe Ratio."""
        daily_returns = np.diff(predicted) / predicted[:-1]
        sharpe_ratio = np.mean(daily_returns) / np.std(daily_returns) * np.sqrt(252)
        return sharpe_ratio

# Helper function to dynamically fetch all pandas-ta indicators and define parameter ranges
def get_indicators_and_ranges():
    """Fetch all indicators from pandas-ta and generate parameter ranges."""
    # Fetch all callable objects in pandas-ta
    indicator_funcs = {
        name: func for name, func in ta.__dict__.items()
        if callable(func) and inspect.isfunction(func)  # Only include proper functions
    }

    param_grids = {}

    for name, func in indicator_funcs.items():
        try:
            # Get the function signature
            sig = inspect.signature(func)
            param_grid = {}

            # Dynamically create parameter ranges based on the function's arguments
            for param in sig.parameters.values():
                if param.name == 'length':
                    param_grid['length'] = [5, 50]  # Example range for lengths
                elif param.name in ['fast', 'slow']:
                    param_grid[param.name] = [5, 50]  # Example for MACD-like indicators
                elif param.name == 'rsi_length':
                    param_grid['rsi_length'] = [5, 30]  # Example for RSI length
                # Add more logic as needed for other parameters

            if param_grid:
                param_grids[name] = param_grid

        except ValueError as e:
            # Skip objects without valid signatures
            print(f"Could not generate parameters for {name}: {e}")
        except Exception as e:
            # Log other unexpected errors but continue
            print(f"Unexpected error for {name}: {e}")
            continue

    return param_grids, indicator_funcs

/Users/habib/Desktop/Projects/FinDRL/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# # Test the function
# param_grids, indicator_funcs = get_indicators_and_ranges()
# print(f"Generated parameter grids for {len(param_grids)} indicators.")

# Example Usage
# if __name__ == "__main__":
#     # Example Data
#     data = pd.DataFrame({
#         'Datetime': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'],
#         'Adj Close': [100, 105, 102, 110, 108],
#         'Volume': [1000, 1200, 1100, 1500, 1400]
#     })
#     data['Datetime'] = pd.to_datetime(data['Datetime'])

#     # Field mapping
#     field_map = {
#         'datetime': 'Datetime',
#         'price': 'Adj Close',
#     }

#     # Get all indicators and parameter ranges
#     param_grids, indicator_funcs = get_indicators_and_ranges()

#     # Example: Optimizing SMA with dynamic parameter range
#     indicator_name = 'sma'
#     if indicator_name in indicator_funcs:
#         optimizer = IndicatorOptimizer(data, field_map, indicator_funcs[indicator_name])
#         best_params, best_performance = optimizer.optimize_parameters(param_grids[indicator_name])

#         print(f"Best Parameters for {indicator_name}: {best_params}")
#         print(f"Best Sharpe Ratio: {best_performance}")


In [4]:
# import pandas as pd
# import numpy as np
# import inspect
# import pandas_ta as ta

# class IndicatorOptimizer:
#     def __init__(self, data):
#         """
#         Initialize the IndicatorOptimizer with data.
        
#         Args:
#         data (pd.DataFrame): The dataset containing OHLCV data and other relevant columns.
#         """
#         self.data = data
#         self.indicators = None
#         self.param_grids = None

#     def get_indicators_and_ranges(self):
#         """
#         Dynamically fetch indicators from pandas-ta and generate parameter ranges.
#         """
#         indicator_funcs = {
#             name: func for name, func in ta.__dict__.items()
#             if callable(func) and inspect.isfunction(func)  # Only include proper functions
#         }
#         param_grids = {}

#         for name, func in indicator_funcs.items():
#             try:
#                 sig = inspect.signature(func)
#                 param_grid = {}

#                 # Dynamically create parameter ranges
#                 for param in sig.parameters.values():
#                     if param.name == 'length':
#                         param_grid['length'] = [5, 50]
#                     elif param.name in ['fast', 'slow']:
#                         param_grid[param.name] = [5, 50]
#                     elif param.name == 'rsi_length':
#                         param_grid['rsi_length'] = [5, 30]
#                 if param_grid:
#                     param_grids[name] = param_grid

#             except Exception as e:
#                 print(f"Could not generate parameters for {name}: {e}")
#                 continue

#         self.indicators = indicator_funcs
#         self.param_grids = param_grids

#     def optimize_parameters(self, metric='sharpe_ratio'):
#         """
#         Optimize the parameters for each indicator using the entire dataset.
        
#         Args:
#         metric (str): Performance metric to optimize. Default is 'sharpe_ratio'.
        
#         Returns:
#         dict: Best parameters for each indicator.
#         """
#         best_params = {}

#         for name, func in self.indicators.items():
#             param_grid = self.param_grids.get(name, {})
#             best_metric = -np.inf
#             best_param_set = {}

#             # Grid search over parameter combinations
#             for param_set in self._generate_param_combinations(param_grid):
#                 try:
#                     # Dynamically pass required columns
#                     required_args = inspect.signature(func).parameters.keys()
#                     kwargs = {arg: self.data[arg.capitalize()] for arg in required_args if arg.lower() in self.data.columns}
#                     kwargs.update(param_set)

#                     indicator_data = func(**kwargs)
#                     if isinstance(indicator_data, pd.DataFrame):
#                         indicator_values = indicator_data.iloc[:, 0]
#                     else:
#                         indicator_values = indicator_data

#                     # Calculate performance metric
#                     performance = self._evaluate_performance(self.data, indicator_values, metric)
#                     if performance > best_metric:
#                         best_metric = performance
#                         best_param_set = param_set
#                 except Exception as e:
#                     print(f"Error optimizing {name} with {param_set}: {e}")

#             best_params[name] = (best_param_set, best_metric)

#         return best_params

#     def _generate_param_combinations(self, param_grid):
#         """
#         Generate all combinations of parameters from the parameter grid.
        
#         Args:
#         param_grid (dict): Dictionary of parameter ranges.
        
#         Yields:
#         dict: Combination of parameters.
#         """
#         from itertools import product

#         keys = param_grid.keys()
#         values = param_grid.values()

#         for combination in product(*values):
#             yield dict(zip(keys, combination))

#     def _evaluate_performance(self, data, indicator_values, metric):
#         """
#         Evaluate the performance of an indicator using the specified metric.
        
#         Args:
#         data (pd.DataFrame): The dataset.
#         indicator_values (pd.Series): Indicator values.
#         metric (str): Metric to evaluate.
        
#         Returns:
#         float: Metric value.
#         """
#         if metric == 'sharpe_ratio':
#             returns = np.log(data['Close'] / data['Close'].shift(1))
#             strategy_returns = returns * np.sign(indicator_values.shift(1))
#             sharpe_ratio = strategy_returns.mean() / strategy_returns.std()
#             return sharpe_ratio
#         else:
#             raise ValueError(f"Unknown metric: {metric}")

#     def update_data_with_indicators(self, best_params):
#         """
#         Update the dataset with calculated indicators using the best parameters.
        
#         Args:
#         best_params (dict): Best parameters for each indicator.
#         """
#         for name, (params, _) in best_params.items():
#             try:
#                 required_args = inspect.signature(self.indicators[name]).parameters.keys()
#                 kwargs = {arg: self.data[arg.capitalize()] for arg in required_args if arg.lower() in self.data.columns}
#                 kwargs.update(params)

#                 indicator_data = self.indicators[name](**kwargs)
#                 if isinstance(indicator_data, pd.DataFrame):
#                     for col_name, series in indicator_data.items():
#                         self.data[f"{name}_{col_name}"] = series
#                 else:
#                     self.data[f"{name}"] = indicator_data
#             except Exception as e:
#                 print(f"Error updating data with {name}: {e}")

# # # Example Usage
# # if __name__ == "__main__":
# #     # Load or create sample data
# #     date_rng = pd.date_range(start='2020-01-01', end='2021-01-01', freq='B')
# #     data = pd.DataFrame({
# #         'Date': date_rng,
# #         'Open': np.random.rand(len(date_rng)) * 100,
# #         'High': np.random.rand(len(date_rng)) * 100,
# #         'Low': np.random.rand(len(date_rng)) * 100,
# #         'Close': np.random.rand(len(date_rng)) * 100,
# #         'Volume': np.random.randint(1, 1000, len(date_rng))
# #     })
# #     data.set_index('Date', inplace=True)

# #     # Initialize optimizer
# #     optimizer = IndicatorOptimizer(data)
# #     optimizer.get_indicators_and_ranges()

# #     # Optimize parameters using the entire dataset
# #     best_params = optimizer.optimize_parameters()

# #     # Update dataset with best indicators
# #     optimizer.update_data_with_indicators(best_params)

# #     print(data.head())


In [5]:
import pandas as pd
import pandas_ta as ta
import inspect
from itertools import product
import numpy as np


class IndicatorOptimizer:
    def __init__(self, data):
        """
        Initializes the IndicatorOptimizer class.
        
        Args:
            data (pd.DataFrame): Financial dataset containing columns like Open, High, Low, Close, Volume.
        """
        self.data = data
        self.indicators = {name: func for name, func in ta.__dict__.items() if callable(func)}
        self.param_grids = self._generate_param_grids()
        self.best_params = {}

    def _generate_param_grids(self):
        """
        Generates parameter grids for each indicator based on their function signature.

        Returns:
            dict: Dictionary with indicator names as keys and parameter ranges as values.
        """
        param_grids = {}
        for name, func in self.indicators.items():
            try:
                signature = inspect.signature(func)
                params = {
                    key: range(2, 51) if "length" in key else [5, 10, 20, 50]  # Default ranges
                    for key, param in signature.parameters.items()
                    if param.default is not param.empty and isinstance(param.default, (int, float))
                }
                param_grids[name] = params
            except Exception as e:
                print(f"Could not generate parameters for {name}: {e}")
        return param_grids

    def _prepare_arguments(self, func):
        """
        Prepares arguments dynamically for a given indicator function.

        Args:
            func (callable): Indicator function.

        Returns:
            dict: Arguments to pass to the indicator function.
        """
        required_args = inspect.signature(func).parameters.keys()
        args = {}
        for arg in required_args:
            col_name = arg.capitalize()
            if col_name in self.data.columns:
                args[arg] = self.data[col_name]
        return args

    def _evaluate_performance(self, data, indicator_values, metric="sharpe"):
        """
        Evaluates the performance of an indicator using a given metric.

        Args:
            data (pd.DataFrame): Financial data.
            indicator_values (pd.Series): Computed indicator values.
            metric (str): Performance metric. Default is "sharpe".

        Returns:
            float: Performance score.
        """
        returns = data["Close"].pct_change()
        if metric == "sharpe":
            # Example: Sharpe ratio calculation
            mean_return = (returns * indicator_values.shift()).mean()
            std_return = (returns * indicator_values.shift()).std()
            return mean_return / std_return if std_return != 0 else -np.inf
        return -np.inf

    def optimize_parameters(self, metric="sharpe"):
        """
        Optimizes the parameters of all indicators for the given dataset.

        Args:
            metric (str): Performance metric to evaluate indicators. Default is "sharpe".

        Returns:
            dict: Best parameters for each indicator.
        """
        for name, func in self.indicators.items():
            param_grid = self.param_grids.get(name, {})
            best_metric = -np.inf
            best_param_set = {}

            for param_set in self._generate_param_combinations(param_grid):
                try:
                    # Dynamically prepare arguments
                    kwargs = self._prepare_arguments(func)
                    kwargs.update(param_set)

                    # Compute the indicator
                    indicator_data = func(**kwargs)
                    indicator_values = (
                        indicator_data.iloc[:, 0] if isinstance(indicator_data, pd.DataFrame) else indicator_data
                    )

                    # Evaluate performance
                    performance = self._evaluate_performance(self.data, indicator_values, metric)
                    if performance > best_metric:
                        best_metric = performance
                        best_param_set = param_set
                except Exception as e:
                    print(f"Error optimizing {name} with {param_set}: {e}")

            self.best_params[name] = (best_param_set, best_metric)
        return self.best_params

    def _generate_param_combinations(self, param_grid):
        """
        Generates all combinations of parameter values for a given grid.

        Args:
            param_grid (dict): Dictionary of parameters and their possible values.

        Returns:
            list: List of parameter combinations as dictionaries.
        """
        keys, values = zip(*param_grid.items()) if param_grid else ([], [])
        return [dict(zip(keys, combination)) for combination in product(*values)]

    def update_data_with_indicators(self):
        """
        Updates the dataset with the best-performing indicators based on optimized parameters.

        Returns:
            pd.DataFrame: Updated dataset with new columns for indicators.
        """
        for name, (params, _) in self.best_params.items():
            try:
                # Prepare arguments and compute indicator
                func = self.indicators[name]
                kwargs = self._prepare_arguments(func)
                kwargs.update(params)

                # Compute and add indicator to data
                indicator_data = func(**kwargs)
                column_name = f"{name}_{'_'.join(f'{k}{v}' for k, v in params.items())}"
                self.data[column_name] = (
                    indicator_data.iloc[:, 0] if isinstance(indicator_data, pd.DataFrame) else indicator_data
                )
            except Exception as e:
                print(f"Error updating data with {name}: {e}")
        return self.data


# # Usage Example
# if __name__ == "__main__":
#     # Example dataset
#     date_rng = pd.date_range(start="2020-01-01", end="2021-01-01", freq="B")
#     data = pd.DataFrame({
#         "Date": date_rng,
#         "Open": np.random.uniform(50, 100, len(date_rng)),
#         "High": np.random.uniform(50, 100, len(date_rng)),
#         "Low": np.random.uniform(50, 100, len(date_rng)),
#         "Close": np.random.uniform(50, 100, len(date_rng)),
#         "Volume": np.random.randint(1000, 10000, len(date_rng)),
#     }).set_index("Date")

#     optimizer = IndicatorOptimizer(data)
#     best_params = optimizer.optimize_parameters()
#     updated_data = optimizer.update_data_with_indicators()

#     print("Best Parameters:", best_params)
#     print("Updated Data:", updated_data.head())


In [35]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class IndicatorOptimizerDL:
    def __init__(self, data, indicators, epochs=100, lr=0.001):
        """
        Initializes the DL-based Indicator Optimizer.

        Args:
            data (pd.DataFrame): Financial dataset containing Open, High, Low, Close, Volume columns.
            indicators (dict): Dictionary of indicator functions with initial parameters (e.g., {"rsi": {"length": 14}}).
            epochs (int): Number of training epochs.
            lr (float): Learning rate for optimizer.
        """
        self.data = data
        self.indicators = indicators
        self.epochs = epochs
        self.lr = lr
        self.scaler = StandardScaler()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _apply_indicators(self, params):
        """
        Apply indicator functions with given parameters to the data.

        Args:
            params (dict): Parameters for each indicator.

        Returns:
            pd.DataFrame: Dataset with indicator features.
        """
        df = self.data.copy()
        for name, (func, param_values) in self.indicators.items():
            # Replace parameters with optimized values
            indicator_values = func(df, **params[name])
            df[f"{name}_optimized"] = indicator_values
        return df

    def _prepare_data(self, df):
        """
        Prepares the dataset for deep learning, scaling and splitting into training and testing sets.

        Args:
            df (pd.DataFrame): Data with indicator features applied.

        Returns:
            tuple: Scaled training and test sets (X_train, X_test, y_train, y_test).
        """
        # Define target as the percentage change in closing price
        df["target"] = df["Close"].pct_change().shift(-1)
        df.dropna(inplace=True)

        # Feature and target split
        X = df.drop(columns=["target"]).values
        y = df["target"].values

        # Standardize the features
        X = self.scaler.fit_transform(X)

        # Split into train and test sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        return X_train, X_test, y_train, y_test

    def _build_model(self, input_dim):
        """
        Builds a simple feedforward neural network.

        Args:
            input_dim (int): Number of input features.

        Returns:
            nn.Module: Neural network model.
        """
        return nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        ).to(self.device)

    def _train_model(self, model, train_loader, optimizer, criterion):
        """
        Trains the neural network model.

        Args:
            model (nn.Module): Neural network model.
            train_loader (DataLoader): DataLoader for training data.
            optimizer (Optimizer): Optimizer for training.
            criterion (Loss): Loss function.

        Returns:
            nn.Module: Trained model.
        """
        model.train()
        for epoch in range(self.epochs):
            epoch_loss = 0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)

                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch.view(-1, 1))
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()

            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch + 1}/{self.epochs}], Loss: {epoch_loss:.4f}")

        return model

    def optimize_indicators(self):
        """
        Optimizes indicator parameters by training a neural network on historical data.

        Returns:
            dict: Optimized parameters for each indicator.
            pd.DataFrame: Updated dataset with optimized indicators applied.
        """
        # Initialize parameters for optimization
        params = {name: torch.tensor(list(param_values.values()), requires_grad=True, dtype=torch.float32, device=self.device)
                  for name, (func, param_values) in self.indicators.items()}

        # Build optimizer for parameters
        optimizer = optim.Adam(params.values(), lr=self.lr)

        # Perform optimization
        for epoch in range(self.epochs):
            optimizer.zero_grad()

            # Apply indicators with current parameters
            applied_data = self._apply_indicators({name: {key: param.item() for key, param in zip(param_values.keys(), param_tensor)}
                                                   for name, (param_tensor, param_values) in zip(params.keys(), self.indicators.items())})

            # Prepare data for training
            X_train, X_test, y_train, y_test = self._prepare_data(applied_data)

            # Build and train model
            train_loader = torch.utils.data.DataLoader(
                dataset=list(zip(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))),
                batch_size=32, shuffle=True
            )
            model = self._build_model(X_train.shape[1])
            criterion = nn.MSELoss()
            self._train_model(model, train_loader, optimizer, criterion)

            # Compute loss on test set
            model.eval()
            with torch.no_grad():
                X_test = torch.tensor(X_test, dtype=torch.float32).to(self.device)
                y_test = torch.tensor(y_test, dtype=torch.float32).to(self.device)
                predictions = model(X_test).cpu().numpy()
                loss = np.mean((predictions - y_test.cpu().numpy())**2)

            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch + 1}/{self.epochs}], Parameter Optimization Loss: {loss:.4f}")

        # Extract optimized parameters
        optimized_params = {name: {key: param.item() for key, param in zip(param_values.keys(), param_tensor)}
                            for name, (param_tensor, param_values) in zip(params.keys(), self.indicators.items())}

        # Return final optimized parameters and updated dataset
        final_data = self._apply_indicators(optimized_params)
        return optimized_params, final_data


# # Example Usage
# if __name__ == "__main__":
#     # Generate synthetic dataset
#     date_rng = pd.date_range(start="2020-01-01", end="2021-01-01", freq="B")
#     data = pd.DataFrame({
#         "Date": date_rng,
#         "Open": np.random.uniform(50, 100, len(date_rng)),
#         "High": np.random.uniform(50, 100, len(date_rng)),
#         "Low": np.random.uniform(50, 100, len(date_rng)),
#         "Close": np.random.uniform(50, 100, len(date_rng)),
#         "Volume": np.random.randint(1000, 10000, len(date_rng)),
#     }).set_index("Date")

#     # Define indicator functions (dummy placeholders here)
#     def rsi(data, length):
#         return data["Close"].rolling(window=int(length)).mean()

#     def sma(data, length):
#         return data["Close"].rolling(window=int(length)).mean()

#     # Initialize the optimizer
#     indicators = {
#         "rsi": (rsi, {"length": 14}),
#         "sma": (sma, {"length": 20}),
#     }
#     optimizer = IndicatorOptimizerDL(data, indicators)

#     # Optimize and update dataset
#     optimized_params, updated_data = optimizer.optimize_indicators()
#     print("Optimized Parameters:", optimized_params)
#     print("Updated Data:", updated_data.head())


In [ ]:
import pandas as pd
import numpy as np
import pandas_ta as ta
import inspect
from typing import Dict, Callable


class DynamicIndicatorOptimizer:
    def __init__(self, data: pd.DataFrame):
        """
        Initialize the optimizer with data and dynamically extract all pandas-ta indicators.

        Args:
            data (pd.DataFrame): Financial dataset containing columns like 'Open', 'High', 'Low', 'Close', and 'Volume'.
        """
        self.data = data
        self.indicators = self._extract_pandas_ta_indicators()

    def _extract_pandas_ta_indicators(self) -> Dict[str, Dict[str, Callable]]:
        """
        Extract all indicators and their default parameters from pandas-ta library.

        Returns:
            dict: A dictionary of indicator functions and their default parameters.
        """
        indicators = {}
        for name, func in vars(ta).items():
            if callable(func) and name[0].islower():  # pandas-ta indicators use lowercase
                try:
                    # Get parameter defaults using inspect.signature
                    sig = inspect.signature(func)
                    params = {
                        param: default.default
                        for param, default in sig.parameters.items()
                        if default.default != inspect.Parameter.empty
                    }
                    indicators[name] = {"function": func, "parameters": params}
                except (ValueError, TypeError):
                    # Skip functions without introspectable parameters
                    continue
        return indicators

    def _apply_indicators(self, params: Dict[str, Dict[str, float]]) -> pd.DataFrame:
        """
        Apply indicators to the dataset with the provided parameters.

        Args:
            params (dict): Parameters for each indicator.

        Returns:
            pd.DataFrame: Dataset with applied indicator columns.
        """
        df = self.data.copy()
        for name, param_set in params.items():
            try:
                # Apply the indicator to the DataFrame
                indicator_values = self.indicators[name]["function"](df, **param_set)
                if isinstance(indicator_values, pd.DataFrame):
                    for col in indicator_values.columns:
                        df[f"{name}_{col}"] = indicator_values[col]
                else:
                    df[f"{name}"] = indicator_values
            except Exception as e:
                print(f"Failed to apply indicator {name}: {e}")
        return df

    def get_indicators_and_parameters(self) -> Dict[str, Dict[str, Callable]]:
        """
        Get a list of available indicators and their default parameters.

        Returns:
            dict: Dictionary of indicators and their parameters.
        """
        return self.indicators

    def optimize_indicators(self):
        """
        Example placeholder for optimizing indicators dynamically.

        Returns:
            dict: Example optimized parameters for indicators.
            pd.DataFrame: Dataset with optimized indicators applied.
        """
        # Example: Define dummy optimization logic
        optimized_params = {
            name: {
                param: value if isinstance(value, (int, float)) else 14  # Use default or fallback
                for param, value in details["parameters"].items()
            }
            for name, details in self.indicators.items()
        }

        # Apply optimized parameters to the dataset
        optimized_data = self._apply_indicators(optimized_params)
        return optimized_params, optimized_data


# Example Usage
if __name__ == "__main__":
    # Generate synthetic dataset
    date_rng = pd.date_range(start="2020-01-01", end="2021-01-01", freq="B")
    data = pd.DataFrame({
        "Date": date_rng,
        "Open": np.random.uniform(50, 100, len(date_rng)),
        "High": np.random.uniform(50, 100, len(date_rng)),
        "Low": np.random.uniform(50, 100, len(date_rng)),
        "Close": np.random.uniform(50, 100, len(date_rng)),
        "Volume": np.random.randint(1000, 10000, len(date_rng)),
    }).set_index("Date")

    # Initialize the optimizer
    optimizer = DynamicIndicatorOptimizer(data)

    # Get all available indicators and their parameters
    indicators = optimizer.get_indicators_and_parameters()
    print("Available Indicators:")
    for name, details in indicators.items():
        print(f"{name}: {details['parameters']}")

    # Optimize indicators (dummy example)
    optimized_params, updated_data = optimizer.optimize_indicators()
    print("\nOptimized Parameters:", optimized_params)
    print("\nUpdated Dataset:")
    print(updated_data.head())


Available Indicators:
find_spec: {'package': None}
get_distribution: {}
volatility: {'tf': 'years', 'returns': False, 'log': False}
dataclass: {'cls': None, 'init': True, 'repr': True, 'eq': True, 'order': False, 'unsafe_hash': False, 'frozen': False, 'match_args': True, 'kw_only': False, 'slots': False, 'weakref_slot': False}
field: {'default': <dataclasses._MISSING_TYPE object at 0x10440a3f0>, 'default_factory': <dataclasses._MISSING_TYPE object at 0x10440a3f0>, 'init': True, 'repr': True, 'hash': None, 'compare': True, 'metadata': None, 'kw_only': <dataclasses._MISSING_TYPE object at 0x10440a3f0>}
cpu_count: {}
simplefilter: {'category': <class 'Warning'>, 'lineno': 0, 'append': False}
npLog10: {}
cdl_doji: {'length': None, 'factor': None, 'scalar': None, 'asint': True, 'offset': None}
cdl_inside: {'asbool': False, 'offset': None}
cdl_pattern: {'name': 'all', 'scalar': None, 'offset': None}
cdl: {'name': 'all', 'scalar': None, 'offset': None}
cdl_z: {'length': None, 'full': None, 'd

/var/folders/4q/r63zv7yj5gz55km0lrg80gwc0000gn/T/ipykernel_3456/4126184382.py:57: DeprecationWarning: bias and ddof have no effect and are deprecated
  indicator_values = self.indicators[name]["function"](df, **param_set)
/Users/habib/Desktop/Projects/FinDRL/.venv/lib/python3.12/site-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: overflow encountered in exp
  result = func(self.values, **kwargs)
/var/folders/4q/r63zv7yj5gz55km0lrg80gwc0000gn/T/ipykernel_3456/4126184382.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  indicator_values = self.indicators[name]["function"](df, **param_set)
/Users/habib/Desktop/Projects/FinDRL/.venv/lib/python3.12/site-packages/numpy/core/_methods.py:53: RuntimeWarning: overflow encountered in reduce
  return umr_prod(a, a

Failed to apply indicator final_time: unsupported operand type(s) for -: 'float' and 'function'
Failed to apply indicator get_time: get_time() got multiple values for argument 'exchange'


/var/folders/4q/r63zv7yj5gz55km0lrg80gwc0000gn/T/ipykernel_3456/4126184382.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{name}_{col}"] = indicator_values[col]
/var/folders/4q/r63zv7yj5gz55km0lrg80gwc0000gn/T/ipykernel_3456/4126184382.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{name}_{col}"] = indicator_values[col]
/var/folders/4q/r63zv7yj5gz55km0lrg80gwc0000gn/T/ipykernel_3456/4126184382.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` ma